# ROGII - Wellbore Geology Prediction — V3 (residual target + smoothing)

Improvement over baseline V1 (public score 17.013): the model no longer learns
`TVT` in absolute terms, but the **residual** `TVT - tvt_extrapolated`. The final
prediction is reconstructed as `tvt_pred = predicted_residual + tvt_extrapolated`.

Rationale: `TVT` is a per-well geological scale, not comparable across wells, so a
model trained directly on it generalizes poorly to unseen wells. The residual is a
relative correction, which transfers much better across wells.

A light post-prediction smoothing (rolling median per well, sorted by MD) is also
applied, since the true TVT curve is geologically continuous.


In [ ]:
# === Imports and Kaggle paths ===
import os, gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

DATA_DIR = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'
train_dir = os.path.join(DATA_DIR, 'train')
test_dir = os.path.join(DATA_DIR, 'test')
OUTPUT_DIR = '/kaggle/working'

train_files = os.listdir(train_dir)
well_names = sorted(set(f.split('__')[0] for f in train_files if '__' in f))
print(f"Number of train wells: {len(well_names)}")

test_files = os.listdir(test_dir)
test_well_names = sorted(set(f.split('__')[0] for f in test_files if '__' in f))
print(f"Number of test wells: {len(test_well_names)}")

In [ ]:
# === GR / typewell signal matching functions ===
WINDOW = 15

def build_typewell_windows(tw_df, window=WINDOW):
    gr = tw_df['GR'].values
    tvt = tw_df['TVT'].values
    n = len(gr)
    if n < window:
        return None, None
    windows = np.lib.stride_tricks.sliding_window_view(gr, window)
    centers = tvt[window // 2 : window // 2 + len(windows)]
    return windows, centers

def vectorized_match_all(hw_gr, tw_windows, tw_centers):
    n = len(hw_gr)
    half = WINDOW // 2
    if n < WINDOW or tw_windows is None:
        return np.full(n, np.nan), np.full(n, np.nan)

    hw_gr = hw_gr.astype(np.float32)
    tw_windows = tw_windows.astype(np.float32)

    hw_windows = np.lib.stride_tricks.sliding_window_view(hw_gr, WINDOW)
    hw_norm = (hw_windows - hw_windows.mean(axis=1, keepdims=True)) / (hw_windows.std(axis=1, keepdims=True) + 1e-6)
    tw_norm = (tw_windows - tw_windows.mean(axis=1, keepdims=True)) / (tw_windows.std(axis=1, keepdims=True) + 1e-6)

    hw_sq = (hw_norm ** 2).sum(axis=1, keepdims=True)
    tw_sq = (tw_norm ** 2).sum(axis=1, keepdims=True).T
    cross = hw_norm @ tw_norm.T
    dists = hw_sq - 2 * cross + tw_sq
    del cross, hw_norm, tw_norm

    best_idx = np.argmin(dists, axis=1)
    best_dist = np.sqrt(np.clip(dists[np.arange(len(dists)), best_idx], 0, None))
    matched_tvt = tw_centers[best_idx]
    del dists

    full_matched = np.full(n, np.nan, dtype=np.float32)
    full_dist = np.full(n, np.nan, dtype=np.float32)
    full_matched[half : half + len(matched_tvt)] = matched_tvt
    full_dist[half : half + len(best_dist)] = best_dist
    full_matched = pd.Series(full_matched).ffill().bfill().values
    full_dist = pd.Series(full_dist).ffill().bfill().values
    return full_matched, full_dist

In [ ]:
# === Feature engineering (target is now the residual) ===
MAX_EXTRAP_DIST = 100  # optimized empirically via CV grid search

def build_features_for_well(hw, tw, is_train=True):
    hw = hw.copy()
    tw_windows, tw_centers = build_typewell_windows(tw, WINDOW)

    matched_tvt, match_dist = vectorized_match_all(hw['GR'].values, tw_windows, tw_centers)
    hw['matched_tvt'] = matched_tvt
    hw['match_dist'] = match_dist

    known_mask = hw['TVT_input'].notna()
    hw['last_known_tvt'] = hw['TVT_input'].ffill()
    hw['last_known_md'] = hw['MD'].where(known_mask).ffill()
    hw['dist_from_anchor'] = hw['MD'] - hw['last_known_md']

    known_idx = np.where(known_mask.values)[0]
    if len(known_idx) >= 10:
        tail = known_idx[-min(100, len(known_idx)):]
        md_tail = hw['MD'].values[tail]
        tvt_tail = hw['TVT_input'].values[tail]
        local_slopes = np.diff(tvt_tail) / np.diff(md_tail)
        slope_tvt = np.median(local_slopes)
    else:
        slope_tvt = 0.0
    hw['slope_tvt'] = slope_tvt

    capped_dist = np.minimum(hw['dist_from_anchor'], MAX_EXTRAP_DIST)
    hw['tvt_extrapolated'] = hw['last_known_tvt'] + slope_tvt * capped_dist

    hw['GR_roll_mean'] = hw['GR'].rolling(WINDOW, center=True, min_periods=1).mean()
    hw['GR_roll_std'] = hw['GR'].rolling(WINDOW, center=True, min_periods=1).std().fillna(0)
    hw['is_known'] = known_mask.astype(int)

    for col in ['X', 'Y', 'Z']:
        diff = hw[col].diff().fillna(0)
        hw[f'{col}_diff_roll'] = diff.rolling(WINDOW, center=True, min_periods=1).std().fillna(0)

    dip_rate = (hw['Z'].diff() / hw['MD'].diff().replace(0, np.nan)).fillna(0)
    hw['dip_rate_roll'] = dip_rate.rolling(WINDOW, center=True, min_periods=1).mean()

    if is_train:
        hw['TVT_true'] = hw['TVT']
        # target = residual, not the absolute value
        hw['target'] = hw['TVT'] - hw['tvt_extrapolated']
    return hw

feature_cols = ['MD', 'X', 'Y', 'Z', 'GR', 'matched_tvt', 'match_dist',
                 'last_known_tvt', 'dist_from_anchor', 'slope_tvt', 'tvt_extrapolated',
                 'GR_roll_mean', 'GR_roll_std', 'is_known',
                 'X_diff_roll', 'Y_diff_roll', 'Z_diff_roll', 'dip_rate_roll']

In [ ]:
# === Build training dataset ===
all_rows = []
for well in tqdm(well_names):
    hw = pd.read_csv(os.path.join(train_dir, f"{well}__horizontal_well.csv"))
    tw = pd.read_csv(os.path.join(train_dir, f"{well}__typewell.csv"))
    if len(hw) < WINDOW or len(tw) < WINDOW:
        continue
    feat = build_features_for_well(hw, tw, is_train=True)
    feat['well'] = well
    keep_cols = feature_cols + ['target', 'TVT_true', 'well']
    small = feat[keep_cols].copy()
    for c in feature_cols + ['target', 'TVT_true']:
        if small[c].dtype == 'float64':
            small[c] = small[c].astype('float32')
    all_rows.append(small)
    del hw, tw, feat, small
    if len(all_rows) % 100 == 0:
        gc.collect()

train_df = pd.concat(all_rows, ignore_index=True)
del all_rows
gc.collect()
print(train_df.shape)

In [ ]:
# === Cross-validation (GroupKFold by well), target = residual ===
X = train_df[feature_cols]
y = train_df['target']              # residual: TVT_true - tvt_extrapolated
y_true = train_df['TVT_true']       # raw TVT, for a score comparable to Kaggle's
groups = train_df['well']

gkf = GroupKFold(n_splits=5)
oof_resid_preds = np.zeros(len(train_df))
models = []
fold_scores = []          # RMSE on reconstructed TVT (comparable to Kaggle score)
fold_scores_naive = []    # RMSE of tvt_extrapolated alone, no model

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
}

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    y_true_val = y_true.iloc[val_idx]

    train_set = lgb.Dataset(X_train, label=y_train)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=2000,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
    )

    resid_pred = model.predict(X_val, num_iteration=model.best_iteration)
    oof_resid_preds[val_idx] = resid_pred

    tvt_pred_val = resid_pred + X_val['tvt_extrapolated'].values
    fold_rmse = np.sqrt(mean_squared_error(y_true_val, tvt_pred_val))
    fold_scores.append(fold_rmse)

    naive_rmse = np.sqrt(mean_squared_error(y_true_val, X_val['tvt_extrapolated'].values))
    fold_scores_naive.append(naive_rmse)

    models.append(model)
    print(f"Fold {fold}: model RMSE = {fold_rmse:.3f}  |  extrapolation-only RMSE = {naive_rmse:.3f}")

oof_tvt_pred = oof_resid_preds + X['tvt_extrapolated'].values
overall_rmse = np.sqrt(mean_squared_error(y_true, oof_tvt_pred))

print(f"\nMean RMSE (model, reconstructed): {np.mean(fold_scores):.3f} (+/- {np.std(fold_scores):.3f})")
print(f"Mean RMSE (extrapolation only):    {np.mean(fold_scores_naive):.3f}")
print(f"Overall OOF RMSE (model):          {overall_rmse:.3f}")

In [ ]:
# === Test several smoothing windows on OOF predictions ===
def smoothed_rmse(pred, train_df, window):
    tmp = train_df[['well', 'MD']].copy()
    tmp['pred'] = pred
    tmp['true'] = train_df['TVT_true'].values
    tmp = tmp.sort_values(['well', 'MD'])
    tmp['pred_smooth'] = tmp.groupby('well')['pred'].transform(
        lambda s: s.rolling(window, center=True, min_periods=1).median()
    )
    return np.sqrt(mean_squared_error(tmp['true'], tmp['pred_smooth']))

windows_to_test = [5, 9, 15, 21, 31, 45, 61, 81]
results = {}
for w in windows_to_test:
    rmse_w = smoothed_rmse(oof_tvt_pred, train_df, w)
    results[w] = rmse_w
    print(f"window={w:>3}: smoothed RMSE = {rmse_w:.3f}")

best_window = min(results, key=results.get)
print(f"\nNo smoothing         : RMSE = {overall_rmse:.3f}")
print(f"Best window ({best_window:>3})   : RMSE = {results[best_window]:.3f}")

SMOOTH_WINDOW = best_window

In [ ]:
# === Retrain final model on 100% of train data (target = residual) ===
final_train_set = lgb.Dataset(X, label=y)
best_iters = [m.best_iteration for m in models]
final_rounds = int(np.mean(best_iters))
print(f"Number of trees for final model: {final_rounds}")

final_model = lgb.train(params, final_train_set, num_boost_round=final_rounds)

In [ ]:
# === Build features on test wells (only rows where TVT_input is NaN) ===
test_rows = []
for well in tqdm(test_well_names):
    hw = pd.read_csv(os.path.join(test_dir, f"{well}__horizontal_well.csv"))
    tw = pd.read_csv(os.path.join(test_dir, f"{well}__typewell.csv"))
    if len(hw) < WINDOW or len(tw) < WINDOW:
        print(f"WARNING: well {well} too short, skipped")
        continue
    hw = hw.reset_index(drop=True)
    hw['row_idx'] = hw.index

    feat = build_features_for_well(hw, tw, is_train=False)
    feat['well'] = well
    feat['row_idx'] = hw['row_idx']

    feat_unknown = feat[hw['TVT_input'].isna()].copy()
    test_rows.append(feat_unknown[feature_cols + ['well', 'row_idx', 'MD']])

test_df = pd.concat(test_rows, ignore_index=True)
print(test_df.shape)

# reconstruct TVT = predicted residual + tvt_extrapolated
resid_pred_test = final_model.predict(test_df[feature_cols])
test_df['tvt_pred'] = resid_pred_test + test_df['tvt_extrapolated'].values

# apply the same smoothing window validated on OOF predictions
test_df_sorted = test_df.sort_values(['well', 'MD'])
test_df_sorted['tvt_pred_smooth'] = test_df_sorted.groupby('well')['tvt_pred'].transform(
    lambda s: s.rolling(SMOOTH_WINDOW, center=True, min_periods=1).median()
)
test_df['tvt_pred_smooth'] = test_df_sorted['tvt_pred_smooth']

test_df['id'] = test_df['well'] + '_' + test_df['row_idx'].astype(str)
submission = test_df[['id', 'tvt_pred_smooth']].rename(columns={'tvt_pred_smooth': 'tvt'})
print(submission.shape)
submission.head(10)

In [ ]:
# === Sanity check against sample_submission.csv ===
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

missing_in_sub = set(submission['id']) - set(sample_sub['id'])
missing_in_sample = set(sample_sub['id']) - set(submission['id'])
print(f"IDs in our submission but not in sample: {len(missing_in_sub)}")
print(f"IDs in sample but not in ours: {len(missing_in_sample)}")
assert len(missing_in_sub) == 0 and len(missing_in_sample) == 0, "ID mismatch with sample_submission.csv!"

print("Any NaN in tvt column:", submission['tvt'].isna().sum())
assert submission['tvt'].isna().sum() == 0, "Submission contains NaN values!" 

In [ ]:
# === Save submission.csv (exact filename required by Kaggle) ===
submission.to_csv(os.path.join(OUTPUT_DIR, 'submission.csv'), index=False)
print("Submission saved to /kaggle/working/submission.csv")

check = pd.read_csv(os.path.join(OUTPUT_DIR, 'submission.csv'))
print(check.shape)
check.head()